# CarRacing: train, plot, run, evaluate and record with SB3
Run after cloning this repository and installing `.[sb3]` in the notebook kernel. The policy below uses PPO directly. Use a new output directory for each training seed. The default budget is an example, not a performance guarantee.

In [ ]:
from pathlib import Path
import sys, json
# Works when the notebook starts in the repository root or in examples/.
EXAMPLES = Path.cwd() if Path("env_utils.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES.resolve()))
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from env_utils import make_env, prepare_env, POLICY, DEFAULT_HISTORY
from plot_rewards import plot_rewards

torch.set_num_threads(1)
run = Path("outputs/notebook_seed0")
run.mkdir(parents=True, exist_ok=True)
assert not any(run.iterdir()), "Use a new run directory."
seed, steps, history = 0, 100_000, DEFAULT_HISTORY
action_repeat = 1  # set to 3 to hold each action for three physics frames


## Train and save
`Monitor` records raw total reward per completed episode. The helper only constructs the environment and observation wrappers; learning is ordinary SB3. No reward normalization is used.

In [ ]:
env = prepare_env(Monitor(make_env(action_repeat=action_repeat), filename=str(run / "training")), history)
env.seed(seed)
try:
    model = PPO(POLICY, env, seed=seed, device="cpu", verbose=1,
                n_steps=512, batch_size=128, n_epochs=4, learning_rate=3e-4)
    model.learn(total_timesteps=steps)
    model.save(str(run / "model"))
    (run / "config.json").write_text(json.dumps({
        "algorithm": "PPO", "policy": POLICY, "history": history, "action_repeat": action_repeat,
        "seed": seed, "actual_timesteps": model.num_timesteps
    }, indent=2))
finally:
    env.close()


## Plot episode returns
The blue curve is the raw return. The orange curve averages the latest 100 completed episodes, using the available episodes at the beginning. Inspect `plot_rewards.py` for the simple CSV/NumPy/Matplotlib implementation.

In [ ]:
plot_rewards(run / "training.monitor.csv", run / "rewards.png")
from IPython.display import Image, display
if (run / "rewards.png").exists():
    display(Image(filename=str(run / "rewards.png")))


## Load and run a model
Evaluation uses deterministic actions and the same saved observation history.

In [ ]:
from env_utils import get_case, episode
config = json.loads((run / "config.json").read_text())
case = get_case("normal_1")
env = prepare_env(make_env(case, action_repeat=config.get("action_repeat", 1)), config["history"])
try:
    loaded = PPO.load(str(run / "model.zip"), env=env, device="cpu")
    print(episode(loaded, env, case["seed"]))
finally:
    env.close()


## Evaluate all ten fixed cases
The manifest is `examples/evaluation_cases.json`. Keep these test cases out of training and model selection. The script saves every episode and group mean/std/success rate in CSV and JSON, including failures. Repeat training with multiple independent training seeds.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(EXAMPLES / "evaluate.py"), "--run", str(run)], check=True)
print((run / "evaluation.json").read_text())


## Make a video
This records every simulation step and the terminal frame. It streams frames to MP4; it does not skip actions. Change `--case` to another ID in the manifest.

In [ ]:
video = run / "demo.mp4"
subprocess.run([sys.executable, str(EXAMPLES / "record_video.py"),
                "--run", str(run), "--case", "normal_1", "--output", str(video)], check=True)
from IPython.display import Video
display(Video(str(video), embed=True))
